# Experiment D: Quantization & Self-Consistency (Kaggle 2x T4 Edition)

This notebook is configured to run Experiment D concurrently on Kaggle's **2x Tesla T4 GPUs**.

### How Kaggle Setup Works:
1. **Accelerator**: Select **GPU T4 x 2** in the right-hand panel.
2. **Internet**: Toggle **Internet on** in the right-hand panel.
3. **Background Execution**: Click **Save Version -> Save & Run All (Commit)**. You can safely close your browser; the job runs for up to 12 hours headlessly.

## 1. Verify 2x T4 GPUs & Environment

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 2. Install Dependencies
Install bitsandbytes and accelerate for quantization.

In [ ]:
!pip install -q --upgrade transformers accelerate bitsandbytes datasets python-dotenv

## 3. Clone Repository / Setup Workspace
If your code is in GitHub, clone it below. Alternatively, upload your project folder as a Kaggle Dataset.

In [ ]:
import os
# If cloning from GitHub:
# !git clone https://github.com/YOUR_USERNAME/Experiment_D.git /kaggle/working/Experiment_D
# %cd /kaggle/working/Experiment_D

# Ensure results directory exists
os.makedirs("/kaggle/working/results", exist_ok=True)
print("Workspace initialized!")

## 4. Hugging Face Login (Required for LLaMA 3.2)
To run LLaMA models, authenticate with your Hugging Face token.
*(You can also use Kaggle Secrets: Add-ons -> Secrets -> HF_TOKEN)*

In [ ]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Successfully authenticated via Kaggle Secrets!")
except Exception as e:
    print("Kaggle Secret not found or error. Enter token manually if running gated models:")
    # login("YOUR_HF_TOKEN_HERE")

## 5. Concurrent Execution on 2x T4 GPUs

Here we run **two independent experiments in parallel**:
- **GPU 0 ()**: Llama-3.2-1B in **16-bit**
- **GPU 1 ()**: Llama-3.2-1B in **8-bit**

Both runs execute simultaneously in the background without fighting for VRAM.

In [ ]:
%%bash
# Run 16-bit on GPU 0
CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python scripts/run_experiment_d.py \n  --model meta-llama/Llama-3.2-1B-Instruct \n  --precision 16bit \n  --dataset gsm8k \n  --n_samples 8 \n  --batch_size 64 \n  --output_dir /kaggle/working/results \n  > gpu0_run.log 2>&1 &

# Run 8-bit on GPU 1 simultaneously
CUDA_VISIBLE_DEVICES=1 PYTHONPATH=. python scripts/run_experiment_d.py \n  --model meta-llama/Llama-3.2-1B-Instruct \n  --precision 8bit \n  --dataset gsm8k \n  --n_samples 8 \n  --batch_size 64 \n  --output_dir /kaggle/working/results \n  > gpu1_run.log 2>&1 &

echo "Launched both runs concurrently! Waiting for completion..."
wait
echo "Both GPU 0 and GPU 1 experiments completed!"

## 6. Live Log Monitoring (Optional)
Run these cells to check the live progress of each GPU while the script is running.

In [ ]:
!tail -n 20 gpu0_run.log

In [ ]:
!tail -n 20 gpu1_run.log

## 7. Run Full Analysis & Generate Plots
Once the runs complete, run  to produce:
1.  with exact timings and accuracies
2. Combined calibration curves
3. Accuracy vs. Precision bar charts

In [ ]:
!PYTHONPATH=. python scripts/analyze_results.py --results_dir /kaggle/working/results

## 8. Display Results & Download Outputs

In [ ]:
import pandas as pd
from IPython.display import Image, display
import glob

df = pd.read_csv("/kaggle/working/results/summary_stats.csv")
display(df)

# Display all generated plots
plots = glob.glob("/kaggle/working/results/*combined*.png") + glob.glob("/kaggle/working/results/*accuracy_vs_precision*.png")
for p in sorted(plots):
    print(f"
=== {p} ===")
    display(Image(filename=p))

In [ ]:
# Zip all results for easy one-click download from Kaggle output tab
!zip -r /kaggle/working/experiment_d_results.zip /kaggle/working/results